# De-identification Cookbook

Short, copy-paste **recipes** for the most common de-identification workflows with OpenMed:

1. De-identify a list or CSV of clinical strings
2. Batch-redact a directory of text files
3. Reversible replace + re-identify round-trip
4. Per-language model selection with `DEFAULT_PII_MODELS`

> **Synthetic data only.** Every name, date, phone number, email, and MRN in this notebook is fabricated. Never run these cells on real PHI you are not authorized to process, and never commit real patient data.

**Prerequisites:** `pip install "openmed[hf]"` (pulls in `transformers`). The recipes use a lightweight 44M English PII model so they stay CPU-friendly. The first call downloads model weights if they are not already cached.

## Setup

Import the public API and pick a lightweight English model. Every recipe reuses these.

In [ ]:
import csv
import tempfile
from pathlib import Path

from openmed import (
    deidentify,
    reidentify,
    BatchProcessor,
    DEFAULT_PII_MODELS,
)

# 44M English PII model - the lightest default, good for CPU-only runs.
MODEL = DEFAULT_PII_MODELS["en"]
print("Using model:", MODEL)

# Shared synthetic notes (fabricated PHI).
NOTES = [
    "Patient John Doe (DOB 01/15/1970) called from 555-123-4567.",
    "Contact Jane Roe at jane.roe@example.com regarding MRN 00123456.",
    "Dr. Alan Grant reviewed the chart for Maria Lopez on 2025-03-02.",
]

## Recipe 1 — De-identify a list or CSV of clinical strings

Call `deidentify()` per string for an in-memory list, then the same idea applied to a CSV column using the stdlib `csv` module (no pandas required).

In [ ]:
# De-identify an in-memory list of strings with the default "mask" method.
for note in NOTES:
    result = deidentify(note, method="mask", model_name=MODEL)
    print(result.deidentified_text)

In [ ]:
# De-identify one column of a CSV and write a redacted copy — stdlib csv only.
workdir = Path(tempfile.mkdtemp(prefix="openmed_cookbook_"))
src_csv = workdir / "patients.csv"
with src_csv.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["id", "note"])
    for i, note in enumerate(NOTES, start=1):
        writer.writerow([i, note])

out_csv = workdir / "patients_deidentified.csv"
with src_csv.open(encoding="utf-8") as fin, out_csv.open("w", newline="", encoding="utf-8") as fout:
    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=reader.fieldnames)
    writer.writeheader()
    for row in reader:
        row["note"] = deidentify(row["note"], method="mask", model_name=MODEL).deidentified_text
        writer.writerow(row)

print(out_csv.read_text(encoding="utf-8"))

## Recipe 2 — Batch-redact a directory with BatchProcessor

`BatchProcessor` with `operation="deidentify"` streams a whole folder of `.txt` files through the model in batches. Use `process_directory()` and write a redacted copy for each successful item.

In [ ]:
# Create a folder of synthetic .txt notes.
notes_dir = workdir / "notes"
notes_dir.mkdir(exist_ok=True)
for i, note in enumerate(NOTES, start=1):
    (notes_dir / f"note_{i}.txt").write_text(note, encoding="utf-8")

processor = BatchProcessor(
    model_name=MODEL,
    operation="deidentify",
    batch_size=8,
    method="mask",
    confidence_threshold=0.7,
)
batch = processor.process_directory(notes_dir, pattern="*.txt")

redacted_dir = workdir / "notes_deidentified"
redacted_dir.mkdir(exist_ok=True)

print(batch.summary())
for item in batch.items:
    if not item.success:
        print(f"Skipped {item.id}: {item.error}")
        continue
    redacted_path = redacted_dir / item.id
    redacted_path.write_text(item.result.deidentified_text, encoding="utf-8")
    print(f"{redacted_path.name}: {redacted_path.read_text(encoding='utf-8')}")

## Recipe 3 — Reversible replace + re-identify round-trip

`method="replace"` swaps PHI for realistic synthetic values. With `keep_mapping=True` you also get a mapping you can feed to `reidentify()` to restore the original — a reversible pipeline for workflows that must re-link later. `consistent=True` keeps the same replacement for repeated entities.

In [ ]:
record = "Patient Sarah Connor, DOB 05/13/1985, phone 555-867-5309, MRN 99887766."

deid = deidentify(
    record,
    method="replace",     # synthetic stand-ins instead of [TAGS]
    model_name=MODEL,
    keep_mapping=True,     # required to reverse later
    consistent=True,       # same entity -> same replacement
)
print("De-identified:", deid.deidentified_text)
print("Mapping:", deid.mapping)

restored = reidentify(deid.deidentified_text, deid.mapping)
print("Re-identified:", restored)

# Round-trip fidelity depends on what the model detected: only entities that were
# replaced can be restored. This is illustrative, not a guarantee for every input.
print("Round-trip matches original:", restored == record)


## Recipe 4 — Per-language model selection with DEFAULT_PII_MODELS

`DEFAULT_PII_MODELS` maps each supported language code to a sensible default PII model. Pass `lang="fr"` (etc.) to `deidentify()`/`extract_pii()` and OpenMed picks the right model and locale-aware patterns automatically. Only the lightweight English model is executed below to avoid heavyweight downloads.

In [ ]:
# Inspect the per-language defaults.
for lang in ["en", "fr", "de", "es", "nl", "hi"]:
    print(f"{lang}: {DEFAULT_PII_MODELS[lang]}")

def model_for(lang: str) -> str:
    """Return the default PII model id for a language code."""
    return DEFAULT_PII_MODELS[lang]

# Run the lightweight English model live:
sample_en = "Patient John Doe was discharged on 2025-01-05."
print(deidentify(sample_en, method="mask", model_name=model_for("en"), lang="en").deidentified_text)

# For other languages, just pass lang=<code>; OpenMed selects the model + locale rules.
# Not executed here to avoid downloading a larger model:
#   deidentify("Né le 15/01/1970 à Paris.", method="mask", lang="fr").deidentified_text

## Summary

You now have four reusable de-identification recipes: list/CSV, directory batch, reversible round-trip, and per-language selection. For the full reference — every method, smart merging, confidence thresholds, custom patterns, and HIPAA notes — see [PII_Detection_Complete_Guide.ipynb](./PII_Detection_Complete_Guide.ipynb) and [Multilingual_PII_Detection_Guide.ipynb](./Multilingual_PII_Detection_Guide.ipynb).

**Remember:** synthetic data only — never commit real PHI.